<a href="https://colab.research.google.com/github/uncannyeng01/yantaramalisonarprojesi_yolo11/blob/main/02_optuna_n2v.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics optuna

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
import os, shutil, re, gc
import optuna, torch
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
OUT  = f'{BASE}/runs_optuna'
DB_LOCAL = '/content/optuna_orijinal.db'
DB_DRIVE = f'{BASE}/optuna_orijinal.db'
os.makedirs(OUT, exist_ok=True)

N_TOPLAM = 12      # toplam deneme (baseline eşdeğeri dahil)
EPOCHS   = 60
PATIENCE = 20
SEED     = 0

if os.path.exists(DB_DRIVE) and not os.path.exists(DB_LOCAL):
    shutil.copy(DB_DRIVE, DB_LOCAL)

def hazirla(ad):
    src, dst = f'{BASE}/dataset_{ad}', f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    if os.path.exists(f'{src}/data.yaml'):
        txt = re.sub(r'^path:.*$', f'path: {dst}', open(f'{src}/data.yaml').read(), flags=re.M)
    else:
        txt = f'path: {dst}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n'
    open(f'{dst}/data.yaml', 'w').write(txt)
    return f'{dst}/data.yaml'

YAML = hazirla('orig')

def objective(trial):
    lr0 = trial.suggest_float('lr0', 5e-4, 5e-3, log=True)
    lrf = trial.suggest_float('lrf', 5e-3, 0.5, log=True)
    wd  = trial.suggest_float('weight_decay', 1e-4, 5e-2, log=True)
    if lr0 == 0.002 and lrf == 0.01 and wd == 0.0005:
        trial.set_user_attr('baseline_esdeger', True)
    name = f'trial{trial.number}'
    YOLO('yolo11s.pt').train(data=YAML, epochs=EPOCHS, patience=PATIENCE, imgsz=960, batch=8,
                             seed=SEED, deterministic=True, workers=2, cache='ram',
                             optimizer='AdamW', momentum=0.9, lr0=lr0, lrf=lrf, weight_decay=wd,
                             project=OUT, name=name, exist_ok=True, plots=False)
    m = YOLO(f'{OUT}/{name}/weights/best.pt')
    r = m.val(data=YAML, split='val', imgsz=960, batch=8, plots=False, verbose=False)   # sadece VAL
    trial.set_user_attr('mAP50', float(r.box.map50))
    gc.collect(); torch.cuda.empty_cache()
    return float(r.box.map)          # val mAP@0.5:0.95

def kaydet(study, trial):
    shutil.copy(DB_LOCAL, DB_DRIVE)

study = optuna.create_study(study_name='orijinal_hp', direction='maximize',
                            storage=f'sqlite:///{DB_LOCAL}', load_if_exists=True,
                            sampler=optuna.samplers.TPESampler(seed=0, n_startup_trials=5))

COMPLETE = optuna.trial.TrialState.COMPLETE
if not any(t.state == COMPLETE and t.user_attrs.get('baseline_esdeger') for t in study.trials):
    study.enqueue_trial({'lr0': 0.002, 'lrf': 0.01, 'weight_decay': 0.0005})   # baseline eşdeğeri

bitmis = len([t for t in study.trials if t.state.is_finished()])
kalan = N_TOPLAM - bitmis
if kalan > 0:
    study.optimize(objective, n_trials=kalan, callbacks=[kaydet])

df = study.trials_dataframe()[['number', 'value', 'params_lr0', 'params_lrf',
                               'params_weight_decay', 'state']]
df.to_csv(f'{BASE}/optuna_orijinal_sonuclar.csv', index=False)
print(df.sort_values('value', ascending=False))
print('EN İYİ:', study.best_trial.params, '| val mAP50-95:', study.best_value)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
import os, shutil, re, gc
import numpy as np, pandas as pd, torch
from ultralytics import YOLO

BASE = '/content/drive/MyDrive/sonar_proje'
OUT  = f'{BASE}/runs_optuna_dogrulama'
os.makedirs(OUT, exist_ok=True)
KARSILASTIR = [0, 6, 7]     # Optuna deneme numaraları
SEEDLER = [1, 2]
EPOCHS, PATIENCE = 60, 20   # Optuna ile aynı bütçe

def hazirla(ad):
    src, dst = f'{BASE}/dataset_{ad}', f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(src, dst)
    if os.path.exists(f'{src}/data.yaml'):
        txt = re.sub(r'^path:.*$', f'path: {dst}', open(f'{src}/data.yaml').read(), flags=re.M)
    else:
        txt = f'path: {dst}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n'
    open(f'{dst}/data.yaml', 'w').write(txt)
    return f'{dst}/data.yaml'

YAML = hazirla('orig')
df = pd.read_csv(f'{BASE}/optuna_orijinal_sonuclar.csv')
kayit = f'{BASE}/optuna_dogrulama_val.csv'
tablo = pd.read_csv(kayit) if os.path.exists(kayit) else pd.DataFrame(
    columns=['deneme', 'seed', 'mAP50_95'])

def var_mi(n, seed):
    return ((tablo.deneme == n) & (tablo.seed == seed)).any()

for n in KARSILASTIR:
    row = df[df.number == n].iloc[0]
    if not var_mi(n, 0):
        tablo = pd.concat([tablo, pd.DataFrame([dict(deneme=n, seed=0, mAP50_95=float(row.value))])])
    for seed in SEEDLER:
        if var_mi(n, seed):
            continue
        name = f'dogrulama_trial{n}_seed{seed}'
        YOLO('yolo11s.pt').train(data=YAML, epochs=EPOCHS, patience=PATIENCE, imgsz=960, batch=8,
                                 seed=seed, deterministic=True, workers=2, cache='ram',
                                 optimizer='AdamW', momentum=0.9, lr0=float(row.params_lr0),
                                 lrf=float(row.params_lrf), weight_decay=float(row.params_weight_decay),
                                 project=OUT, name=name, exist_ok=True, plots=False)
        m = YOLO(f'{OUT}/{name}/weights/best.pt')
        r = m.val(data=YAML, split='val', imgsz=960, batch=8, plots=False, verbose=False)
        tablo = pd.concat([tablo, pd.DataFrame([dict(deneme=n, seed=seed, mAP50_95=float(r.box.map))])])
        tablo.to_csv(kayit, index=False)
        gc.collect(); torch.cuda.empty_cache()
tablo.to_csv(kayit, index=False)
print(tablo.sort_values(['deneme', 'seed']))
print(tablo.groupby('deneme').mAP50_95.agg(['mean', 'std']))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
BASE = '/content/drive/MyDrive/sonar_proje'
print('sonar_proje var mı:', os.path.exists(BASE))
print('dataset_n2v var mı:', os.path.exists(f'{BASE}/dataset_n2v'))
print('dataset_n2v_struct var mı:', os.path.exists(f'{BASE}/dataset_n2v_struct'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

import os, gc, shutil, torch, pandas as pd
from ultralytics import YOLO
assert torch.cuda.is_available(), 'GPU yok -> T4 GPU seç ve tekrar çalıştır'
gpu = torch.cuda.get_device_name(0); print('GPU:', gpu)
if 'T4' not in gpu: print('UYARI: önceki çalışmalar T4 idi, tutarlılık için T4 tercih et')

BASE = '/content/drive/MyDrive/sonar_proje'
DENEYLER = ['n2v']
SEEDLER = [0, 1, 2]
AUTO_DISCONNECT = True
OUT = f'{BASE}/runs_n2v'; os.makedirs(OUT, exist_ok=True)

def hazirla(ad):
    dst = f'/content/dataset_{ad}'
    if not os.path.exists(dst):
        shutil.copytree(f'{BASE}/dataset_{ad}', dst)
    open(f'{dst}/data.yaml', 'w').write(
        f"path: {dst}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n")
    print(ad, {s: len(os.listdir(f'{dst}/images/{s}')) for s in ['train', 'val', 'test']})
    return f'{dst}/data.yaml'

def bir_deney(ad, seed, yaml_yol):
    name = f'seed{seed}_{ad}'
    done = f'{OUT}/{name}.done'
    if os.path.exists(done):
        print('atlandı (bitmiş):', name); return
    rdir = f'{OUT}/{name}'
    best, last = f'{rdir}/weights/best.pt', f'{rdir}/weights/last.pt'
    print(f'\n=== {name} başlıyor ===')
    if os.path.exists(last):
        try:
            YOLO(last).train(resume=True)
        except Exception as e:
            print('devam edilemedi (muhtemelen zaten bitmiş):', str(e)[:150])
    else:
        YOLO('yolo11s.pt').train(data=yaml_yol, epochs=100, patience=30, imgsz=960, batch=8,
                                 seed=seed, deterministic=True, workers=2, cache='ram',
                                 project=OUT, name=name, exist_ok=True, plots=False)
    m = YOLO(best); rows = []
    for split in ['val', 'test']:
        r = m.val(data=yaml_yol, split=split, imgsz=960, batch=8, verbose=False, plots=False,
                  project=OUT, name=f'eval_{name}_{split}', exist_ok=True)
        P, R = r.box.mp, r.box.mr
        rows.append(dict(deney=ad, seed=seed, kume=split, precision=P, recall=R,
                         f1=2*P*R/(P+R) if P+R > 0 else 0, mAP50=r.box.map50, mAP50_95=r.box.map))
    csv = f'{BASE}/results_{ad}.csv'
    new = pd.DataFrame(rows).round(4)
    if os.path.exists(csv):
        new = pd.concat([pd.read_csv(csv), new])
    new.drop_duplicates(['deney', 'seed', 'kume'], keep='last').to_csv(csv, index=False)
    open(done, 'w').write('ok')
    del m; gc.collect(); torch.cuda.empty_cache()

for ad in DENEYLER:
    yaml_yol = hazirla(ad)
    for seed in SEEDLER:
        bir_deney(ad, seed, yaml_yol)

cols = ['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']
for ad in DENEYLER:
    d0 = pd.read_csv(f'{BASE}/results_{ad}.csv')
    for kume in ['val', 'test']:
        d = d0[d0.kume == kume]
        print(f'\n== {ad} | {kume} | n={len(d)} ==')
        print(' | '.join(f'{c} {d[c].mean():.3f}±{d[c].std():.3f}' for c in cols))

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()